# Drone vs bird on real images first, then fine-tune on Gazebo

Two stages (the plan's "real first, then Gazebo"):
1. **Stage 1:** YOLO26 (COCO weights) learns drone vs bird on ~21 000 real images ([Drone vs Bird, Mendeley](https://data.mendeley.com/datasets/6ghdz52pd7/2), CC BY 4.0: credit it in the report).
2. **Stage 2:** that model is fine-tuned on our Gazebo images (`gen.zip`, or the cut-outs `crops.zip`), then scored by pixel size on our Gazebo test run.
3. **Optional comparison:** the same stage 2 straight from COCO weights (Gazebo only), to see whether stage 1 helped.

Runtime → Change runtime type → **GPU** first. Then set the settings in the next cell and run the cells in order. See `training/README.md` in the repo.

In [ ]:
# ---- Settings ----------------------------------------------------------------------
MODEL = 'yolo26n.pt'      # starting weights: yolo26n.pt (small, fast) or yolo26s.pt (bigger, better, slower)
BATCH = 16                # lower if the GPU runs out of memory (T4: 16 at 640 px, 8 at 1280 px)

# Stage 1: real drone-vs-bird images (640x640)
STAGE1_EPOCHS = 30
STAGE1_FREEZE = None      # None = train all layers (best: the drone class is new to the COCO weights)

# Stage 2: our Gazebo images
STAGE2_DATA = 'full'      # 'full' = gen.zip (full images, 1280 px) or 'crops' = crops.zip (cut-outs, 128 px)
STAGE2_EPOCHS = 50
STAGE2_FREEZE = 10        # freeze layers 0..9: the backbone keeps what stage 1 learned; None = train everything
STAGE2_LR0 = 0.002        # lower than a fresh training (~0.01): small steps so stage 1 is not overwritten

COMPARE_GAZEBO_ONLY = True   # also train stage 2 from plain COCO weights, for comparison

STAGE2_IMGSZ = 1280 if STAGE2_DATA == 'full' else 128
def opt(flag, value):
    return '' if value is None else f'{flag} {value}'

In [ ]:
# 1. GPU, Ultralytics, and the files from the Mac: train_yolo.py and gen.zip (or crops.zip)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics
from google.colab import files
uploaded = files.upload()   # pick train_yolo.py and gen.zip / crops.zip (several at once is fine)
print('Uploaded:', list(uploaded))

In [ ]:
# 2. Download the real drone-vs-bird set (~1 GB) straight into Colab
!wget -q -O dvb.zip "https://data.mendeley.com/public-api/zip/6ghdz52pd7/download/2"
!rm -rf dvb && mkdir -p dvb && unzip -q -o dvb.zip -d dvb
# Mendeley sometimes wraps the dataset in another zip: unpack those too
import glob, subprocess
for inner in glob.glob('dvb/**/*.zip', recursive=True):
    subprocess.run(['unzip', '-q', '-o', inner, '-d', inner[:-4]], check=True)
!find dvb -maxdepth 4 -name '*.yaml' -o -maxdepth 4 -type d -name labels | head -20

In [ ]:
# 3. Put it in our class order (0 drone, 1 bird)
# The class is taken from the file name: in this set bird images are called "BTR (...)",
# "BVA (...)" etc. and drone images "DTR (...)", "DVA (...)": the first letter says it.
# (The class numbers inside the label files can't be trusted: on 18 Oct they all came out
# as one class, so every bird was trained as a drone.) The data.yaml names are printed
# only for reference.
import os, re
from pathlib import Path
from collections import Counter
for y in sorted(glob.glob('dvb/**/data.yaml', recursive=True))[:1]:
    print(y, '\n', Path(y).read_text())

def class_from_name(name):
    first = name.strip().upper()[:1]
    return 0 if first == 'D' else 1 if first == 'B' else None

out = Path('/content/dvb_ours')
os.system(f'rm -rf {out}')
counts, classes, skipped = Counter(), Counter(), Counter()
for split, ours in (('train', 'train'), ('valid', 'val'), ('val', 'val'), ('test', 'test')):
    for labels in glob.glob(f'dvb/**/{split}/labels', recursive=True):
        images = Path(labels).parent / 'images'
        (out / ours / 'images').mkdir(parents=True, exist_ok=True)
        (out / ours / 'labels').mkdir(parents=True, exist_ok=True)
        for lab in Path(labels).glob('*.txt'):
            cls = class_from_name(lab.name)
            if cls is None:
                skipped[lab.name[:3]] += 1
                continue
            lines = []
            for line in lab.read_text().splitlines():
                parts = line.split()
                if len(parts) >= 5:
                    xy = list(map(float, parts[1:]))
                    if len(xy) > 4:   # segmentation polygon -> box
                        xs, ys = xy[0::2], xy[1::2]
                        xy = [(min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)]
                    lines.append(f'{cls} ' + ' '.join(f'{v:.6f}' for v in xy[:4]))
                    classes[(ours, cls)] += 1
            (out / ours / 'labels' / lab.name).write_text('\n'.join(lines) + ('\n' if lines else ''))
            for ext in ('.jpg', '.jpeg', '.png'):
                img = images / (lab.stem + ext)
                if img.exists():
                    dst = out / ours / 'images' / img.name
                    if not dst.exists():
                        os.symlink(img.resolve(), dst)
                    counts[ours] += 1
                    break
print('images per split:', dict(counts))
print('boxes per split and class (0 drone, 1 bird):', dict(sorted(classes.items())))
if skipped:
    print('skipped (name does not start with B or D):', dict(skipped))
assert all(classes[(s, c)] > 0 for s in ('train', 'val') for c in (0, 1)), 'both classes must be in train and val'
(out / 'data.yaml').write_text(f'path: {out}\ntrain: train/images\nval: val/images\n'
                               f'test: test/images\nnames:\n  0: drone\n  1: bird\n')

In [ ]:
# 4. Stage 1: drone vs bird on the real images
# --mosaic 0: these images are already 2x2 mosaics (made by Roboflow), so no second mosaic
!python3 train_yolo.py --data /content/dvb_ours/data.yaml --model {MODEL} --imgsz 640 \
    --epochs {STAGE1_EPOCHS} --batch {BATCH} --device 0 --name stage1_dvb --no-score --mosaic 0 {opt('--freeze', STAGE1_FREEZE)}
STAGE1 = sorted(glob.glob('runs/stage1_dvb*/weights/best.pt'), key=os.path.getmtime)[-1]
print('Stage 1 weights:', STAGE1)

In [ ]:
# 5. Stage 2: fine-tune on our Gazebo images, then score by pixel size on our test run
ZIP = 'gen.zip' if STAGE2_DATA == 'full' else 'crops.zip'
FOLDER = 'gen' if STAGE2_DATA == 'full' else 'crops'
!unzip -q -o {ZIP}
!python3 train_yolo.py --data {FOLDER}/data.yaml --model {STAGE1} --imgsz {STAGE2_IMGSZ} \
    --epochs {STAGE2_EPOCHS} --batch {BATCH} --device 0 --name stage2_from_dvb \
    {opt('--freeze', STAGE2_FREEZE)} {opt('--lr0', STAGE2_LR0)}

In [ ]:
# 6. Optional: the same stage 2 from plain COCO weights (Gazebo only), to compare with the table above
if COMPARE_GAZEBO_ONLY:
    !python3 train_yolo.py --data {FOLDER}/data.yaml --model {MODEL} --imgsz {STAGE2_IMGSZ} \
        --epochs {STAGE2_EPOCHS} --batch {BATCH} --device 0 --name stage2_gazebo_only

In [ ]:
# 7. Export the fine-tuned model to ONNX and download everything in one zip
import shutil
from ultralytics import YOLO
run = sorted(glob.glob('runs/stage2_from_dvb*/'), key=os.path.getmtime)[-1]
YOLO(run + 'weights/best.pt').export(format='onnx', imgsz=STAGE2_IMGSZ, opset=12)
for r in glob.glob('runs/*/weights/last.pt'):
    os.remove(r)   # keep the zip small: best.pt is enough
shutil.make_archive('yolo_dvb_results', 'zip', 'runs')
files.download('yolo_dvb_results.zip')

## Settings explained

| Setting | What it does | Suggestion |
|---|---|---|
| `MODEL` | Starting weights. `n` (2.5 M parameters) is fast and what runs on the drone; `s` (~10 M) is more accurate, for the base station | `yolo26n.pt` |
| `STAGE1_FREEZE` | Layers kept fixed in stage 1. COCO has no drone class, so letting everything learn is best | `None` |
| `STAGE2_FREEZE` | Layers kept fixed in stage 2. YOLO26: layers 0–10 are the **backbone** (general features: edges, shapes), 11–22 the **neck** (combines scales), 23 the **head** (boxes and classes). `10` keeps the backbone from stage 1 and adapts neck + head to Gazebo; `23` trains only the head (fastest, least change); `None` trains everything (most change; best with lots of Gazebo data, risks forgetting stage 1) | `10` |
| `STAGE2_LR0` | Step size. Fine-tuning uses a smaller one than training from scratch, so stage 1's knowledge is adjusted rather than overwritten | `0.002` (try 0.001–0.005) |
| `*_EPOCHS` | Passes over the data. Early stopping (15 epochs without val improvement) stops sooner if nothing improves | 30 / 50 |
| `BATCH` | Images per step. Bigger is steadier but needs more GPU memory | 16 at 640 px, 8 at 1280 px on a T4 |
| `STAGE2_DATA` | `full`: full Gazebo frames at 1280 px. `crops`: the cut-outs from `makecrops` at 128 px (the two-stage approach) | try both |

Fixed in `train_yolo.py` for both stages: `scale=0.2` (less shrinking, so tiny drones survive augmentation) and stronger brightness changes (`hsv_v=0.5`).